In [24]:
!nvidia-smi

Fri May 10 16:31:07 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla K80           Off  | 00000000:05:00.0 Off |                    0 |
| N/A   41C    P8    58W / 149W |     53MiB / 11441MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  Tesla K80           Off  | 00000000:06:00.0 Off |                    0 |
| N/A   

In [ ]:
import os
import nibabel as nib
import numpy as np 
from collections import OrderedDict
import json
from pathlib import Path

from utils.helper import convert_nrrd_to_nifti, create_folder, plot_all_slices, plot_histogram, generate_binary_image, adjust_affine_for_spacing_and_origin, save_binary_image_with_adjusted_origin, make_if_dont_exist,load_nifti_file,convert_file_format,file_exists, load_nifti_file_af_datatype
from utils.metrics import dice_score_per_class, hausdorff_distance_per_class, ravd_per_class

In [ ]:

# define dataset path
BASE_PATH = Path('./').resolve()
DATA_PATH = BASE_PATH / 'dataset'

project_name = 'HCFC1' #change here for different task name
task_name = 'Dataset002_' + project_name 

TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTr'
GT_TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTr'
TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTs'
GT_TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTs'
PREDICTION_RESULTS_PATH  = BASE_PATH / 'dataset/nnUNet_Prediction_Results' / task_name
TASK_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name 

# setup environment variables
nnUNet_raw = BASE_PATH / 'dataset/nnUNet_raw_data'
nnUNet_preprocessed = BASE_PATH / 'dataset/nnUNet_preprocessed'
nnUNet_results = BASE_PATH / 'dataset/nnUNet_results'

In [ ]:
%env nnUNet_raw=$nnUNet_raw
%env nnUNet_preprocessed=$nnUNet_preprocessed
%env nnUNet_results=$nnUNet_results

env: nnUNet_raw=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data
env: nnUNet_preprocessed=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_preprocessed
env: nnUNet_results=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results


In [ ]:
!nnUNetv2_predict -d Dataset001_Wdr47Kusss -i /work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/ -o /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/binary/ -f 0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans

In [5]:
#try with nifti format data. 

source_path="/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/"
binary_path="/work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/BINARY-TEST/"
save_path="/work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/TEST-VOLUME/" 

multiply_vol_save_nifti(source_path, binary_path, save_path)

# endswith = '.nrrd'
# convert_file_format(save_path,save_path,endswith)

!ls /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/TEST-VOLUME/



/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/NG2613_right_0000.nii.gz
Saved successfully: /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/TEST-VOLUME/NG2613_right.nii.gz
/work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/NG2614_left_0000.nii.gz
Saved successfully: /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/TEST-VOLUME/NG2614_left.nii.gz
NG2613_right.nii.gz  NG2614_left.nii.gz


In [7]:
!nnUNetv2_predict -d Dataset002_HCFC1 -i /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/TEST-VOLUME/ -o /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/PRED-VOLUME/ -f  0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 2 cases in the source folder
[[], []]
Traceback (most recent call last):
  File "/user1/ngmm/tr855969/.conda/envs/env_tf/bin/nnUNetv2_predict", line 8, in <module>
    sys.exit(predict_entry_point())
  File "/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 867, in predict_entry_point
    predictor.predict_from_files(args.i, args.o, save_probabilities=args.save_probabilities,
  File "/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 247, in predict_fr